This notebook is to run inference on a discrete state and action space environment for UAV path planning. The environment parameters should be in the "exp_sets" folder and the CoppeliaSim environments should be in the "coppeliasim_envs" folder. The trained models should be in the "trained_models" folder.

In [5]:
# Hyperparameters and paths
exp_alg = 'a2c'
exp_set = 'set1'
num_robots = 3
json_path = rf'..\exp_sets\uav\dis_sets.json'
trained_model_path = rf"..\trained_models\uav\dis_env1_3robots_A2C.zip"

In [6]:
# Import libraries
import json
import random
import numpy as np
import pygame
import gymnasium as gym
from gymnasium import spaces
from shapely import Polygon
from shapely.geometry import Point
np.random.seed(33) # seeding

from coppeliasim_zmqremoteapi_client import RemoteAPIClient
if exp_alg == 'a2c':
    from stable_baselines3 import A2C as model_loader # For PPO, need to change this

In [7]:
# function for random starting points
def random_starting_locations(points, infected_locations):
    points = set([tuple(p) for p in points])
    st_locs = points.difference(infected_locations) # for points - infected_locations
    st_locs = random.choices(np.array(list(st_locs)),k=3)
    return st_locs

# convert binary list to decimal
def binary_list_to_decimal(bin_list):
    bin = ''
    for b in bin_list:
        bin += str(b)
    dec = int(bin,2)
    return dec

# Load experiment json file
def load_experiment_dict_json(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    for set_name, cfg in data.items():
        # Convert field to list of tuples
        cfg["field"] = [tuple(p) for p in cfg["field"]]
        # Convert init_positions to NumPy arrays
        cfg["init_positions"] = [np.array(p, dtype=float) for p in cfg["init_positions"]]
        # Convert infected_locations to set of tuples
        cfg["infected_locations"] = {tuple(p) for p in cfg["infected_locations"]}
    return data

In [8]:
selected_experiment = load_experiment_dict_json(json_path)[exp_set]
selected_experiment

{'field': [(43, 24), (20, 47), (13, 34), (12, 9), (22, 0)],
 'init_positions': [array([14., 34.]), array([40., 25.]), array([20., 40.])],
 'infected_locations': {(15, 10),
  (22, 13),
  (25, 20),
  (25, 35),
  (26, 21),
  (35, 25)}}

Make the multi-agent class:

In [9]:
# Cell 4
class ThreeAgentGridworldEnv(gym.Env):
    metadata = {'render_modes': ['human', 'print', 'rgb_array'], "render_fps": 4}    
    def __init__(self, render_mode=None, grid_size=(50, 50), poly_vertices=selected_experiment['field']):
        super(ThreeAgentGridworldEnv, self).__init__()
        self.Poly = Polygon(poly_vertices) # Get the points of the polygon
        self.poly_vertices = poly_vertices # Vertices of polygon
        self.size = grid_size[0]  # The size of the square grid
        self.window_size = 800  # The size of the PyGame window        
        self.grid_size = grid_size # Size of the grid (May need to remove later on)
        self.outer_boundary = self.Poly.buffer(distance=2) # outer boundary with buffer of distance 2

        # Observation points
        self.observation_points = self.obs_points()
        # print('Observation points:', self.observation_points)
        self.observation_length = len(self.observation_points)
        self.observation_map = {tuple(v):i for i,v in enumerate(self.observation_points)}
        self.step_count = 0
        self.visited = set()
        self.infected_locations = selected_experiment['infected_locations']
        
        self.infected_length = len(self.infected_locations)
        self.infected_state_length = 2**(len(self.infected_locations)) # 2**5, binary to decimal
        self.infected_dict = {v:0 for v in self.infected_locations} # dictionary of locations
        
        # Action and observation space
        self.action_space = spaces.MultiDiscrete([5, 5, 5])  # 4 possible actions for each of the two agents
        self.observation_space = spaces.MultiDiscrete([self.observation_length, self.observation_length, self.observation_length, self.infected_state_length])
        
        assert render_mode is None or render_mode in self.metadata["render_modes"] # Check if the render mode is correct
        self.render_mode = render_mode
        # If human-rendering is used, `self.window` will be a reference
        # to the window that we draw to. `self.clock` will be a clock that is used
        # to ensure that the environment is rendered at the correct framerate in
        # human-mode. They will remain `None` until human-mode is used for the
        # first time.
        self.window = None
        self.clock = None

        # Reset the environment and start
        self.reset()

    def obs_points(self):
        xp,yp = self.Poly.exterior.xy
        # Gridpoints
        xs = np.arange(0, 90, 1)
        ys = np.arange(0, 90, 1)
        # Inside points
        # print(xs,ys)
        xps, yps = [], []
        for xi in xs:
            for yi in ys:
                p = Point(xi,yi)
                if self.Poly.contains(p):
                    xps += [p.x]
                    yps += [p.y]
        xps += xp
        yps += yp
        # plt.plot(xp,yp)
        # plt.scatter(xps,yps, color='r')
        obs_points = np.array(list(set(zip(xps,yps)))) # Taking unique observation points
        return obs_points
    
    def _get_obs(self):
        a1, a2, a3 = self.agent_positions[0], self.agent_positions[1], self.agent_positions[2]
        info = {'agent1': a1, 'agent2': a2, 'agent3': a3, 'step_count': self.step_count}
        # print('agent positions:', a1, a2, a3)
        p1,p2,p3 = self.observation_map[tuple(a1)], self.observation_map[tuple(a2)], self.observation_map[tuple(a3)]
        infected = binary_list_to_decimal(list(self.infected_dict.values()))
        state = np.array([p1,p2,p3,infected]) # convert the infected binary list to decimal
        return state, info

    def reset(self, seed=None, options={}):
        self.visited = set()
        self.step_count = 0
        self.infected_locations = {(15,10), (22,13), (25,20), (26,21), (35,25), (25, 35)}
        self.infected_dict = {v:0 for v in self.infected_locations} # dictionary of locations
        self.agent_positions = selected_experiment['init_positions']
        return self._get_obs()

    def step(self, action):
        # Placeholder for terminal state and rewards
        terminated, truncated = False, False
        rewards = 0
        self.step_count += 1

        # Define the movements corresponding to each action
        movements = [(-1, 0), (1, 0), (0, -1), (0, 1), (0, 0)]  # up, down, left, right, none
        
        # Update the positions of both agents
        for i, act in enumerate(action):
            # if act == 4: # If the action is none
            #     terminated = True # Terminated
            #     break

            movement = movements[act] # What movement to take
            new_position = self.agent_positions[i] + movement # New position after movement
            
            # Ensure the new position is within bounds
            # new_position = np.clip(new_position, [0, 0], [self.grid_size[0]-1, self.grid_size[1]-1])
            new_p = Point(new_position[0], new_position[1])
            # print("checking new position", tuple(new_position))
            if self.Poly.contains(new_p):
                self.agent_positions[i] = new_position
            else:
                rewards -= 10
            if tuple(new_position) in self.visited:
                rewards -= 10
            else:
                rewards -= 1
            self.visited.add(tuple(new_position))
        
        # Check if an infected location is visited
        infected_visited = [x for x in self.agent_positions if tuple(x) in self.infected_locations] # If infected cells are visited
        for v in infected_visited:
            if tuple(v) in self.infected_locations:
                self.infected_locations.remove(tuple(v))
                self.infected_dict[tuple(v)] = 1
        
        if infected_visited:
            rewards += 100 * len(infected_visited) 

        if sum(list(self.infected_dict.values()))==self.infected_length:
            rewards += 100000
            terminated = True

        
        # If the agents meet at the same position, we can assign a reward or consider it a terminal state
        if np.array_equal(self.agent_positions[0], self.agent_positions[1]) or np.array_equal(self.agent_positions[0], self.agent_positions[2]) or np.array_equal(self.agent_positions[1], self.agent_positions[2]):
            rewards -= 100000  # Infinity reward for meeting at the same position
            terminated = True
        
        obs, info = self._get_obs()
        # rewards = rewards * self.gamma ** self.step_count
        return obs, rewards, terminated, truncated, info

    def render(self):
        if self.render_mode == 'print':
            grid = np.zeros(self.grid_size)
            grid[tuple(self.agent_positions[0])] = 1  # Mark the position of the first agent
            grid[tuple(self.agent_positions[1])] = 2  # Mark the position of the second agent
            print(grid)
        else:
            if self.window is None and self.render_mode == "human": # Initialize pygame if it is not initialized
                pygame.init()
                pygame.display.init()
                self.window = pygame.display.set_mode(
                    (self.window_size, self.window_size)
                )
            if self.clock is None and self.render_mode == "human":
                self.clock = pygame.time.Clock()
            
            # Fill the canvas
            canvas = pygame.Surface((self.window_size, self.window_size))
            canvas.fill((255, 255, 255))
            pix_square_size = (
                self.window_size / self.size
            )  # The size of a single grid square in pixels

            # Draw the polygon
            pixel_poly_vertices = [(point[0] * pix_square_size, point[1] * pix_square_size) for point in self.poly_vertices]
            pygame.draw.polygon(surface=canvas, 
                                color=(255, 255, 0), 
                                points=pixel_poly_vertices)
            
            # Draw the visited regions
            for p in self.visited:
                pygame.draw.rect(
                canvas,
                pygame.Color(100, 100, 100, a=0.5),
                pygame.Rect(
                    pix_square_size * np.array(p),
                    (pix_square_size, pix_square_size),
                ),
                )
            # Draw agent1 (square)
            pygame.draw.rect(
                canvas,
                (255, 0, 0),
                pygame.Rect(
                    pix_square_size * self.agent_positions[0],
                    (pix_square_size, pix_square_size),
                ),
            )
            # Draw agent2 (circle)
            pygame.draw.circle(
                canvas,
                (0, 0, 255),
                (self.agent_positions[1] + 0.5) * pix_square_size,
                pix_square_size / 3,
            )
            # Draw agent3 (circle)
            pygame.draw.circle(
                canvas,
                (0, 255, 0),
                (self.agent_positions[2] + 0.5) * pix_square_size,
                pix_square_size / 3,
            )
            # Draw infected locations
            for l in self.infected_locations:
                pygame.draw.rect(
                    canvas,
                    (0, 255, 255),
                    pygame.Rect(
                        pix_square_size * np.array(l),
                        (pix_square_size, pix_square_size),
                    ),
                )


            if self.render_mode == "human":
                # The following line copies our drawings from `canvas` to the visible window
                self.window.blit(canvas, canvas.get_rect())
                pygame.event.pump()
                pygame.display.update()

                # We need to ensure that human-rendering occurs at the predefined framerate.
                # The following line will automatically add a delay to keep the framerate stable.
                self.clock.tick(self.metadata["render_fps"])
                # Finally
                pygame.event.get()

            elif self.render_mode == 'rgb_array':  # rgb_array
                return np.transpose(
                    np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2)
                )
            
    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()

# Register the environment
gym.envs.registration.register(
    id='ThreeAgentGridworld-v0',
    entry_point=ThreeAgentGridworldEnv,
    max_episode_steps=2000,
)

# Inference

Load trained network:

In [10]:
# Load trained network
model = model_loader.load(trained_model_path)
assert False, "open the scene file using CoppeliaSim robot simulator before running the below cells"

c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


AssertionError: open the scene file using CoppeliaSim robot simulator before running the below cells

## Simulation

**IMPORTANT**: open the scene file using CoppeliaSim robot simulator before running the below cells

In [11]:
# Start the remote API client
client = RemoteAPIClient()
sim = client.getObject('sim')
defaultIdleFps = sim.getInt32Param(sim.intparam_idle_fps)
sim.setInt32Param(sim.intparam_idle_fps, 0)

# Height of movement
height = 0.35

class Drone_simulator:    
    def __init__(self, polygon, scaling_factor, height):
        self.scaling_factor = scaling_factor
        self.scaled_polygon = [(x/scaling_factor,y/scaling_factor) for (x,y) in polygon]
        self.rounded_polygon = self.scaled_polygon + [self.scaled_polygon[0]]
        self.color = [[255,0,0],[255,0,255],[0,0,255]]
        self.edges_3d = self.calc_edges_3d()
        self.height = height

    def start_simulation(self):
        self.trace_line = sim.addDrawingObject(sim.drawing_lines, 2, 0, -1, 9999, [255,0,0]) # red line
        sim.startSimulation()
        print('Program started')

    def stop_simulation(self):
        sim.removeDrawingObject(self.trace_line)
        sim.stopSimulation()

    def calc_edges_3d(self):  # To calculate the edges in the polygon
        edges = []
        for i in range(len(self.rounded_polygon) - 1):
            edges.append([list(self.rounded_polygon[i]), list(self.rounded_polygon[i+1])])
        return edges

    def draw_field(self):
        white = [255, 255, 255]
        lineContainer = sim.addDrawingObject(sim.drawing_lines, 2, 0, -1, 9999, white)
        for l in self.edges_3d: # Drawing the field with white lines
            line = l[0] + [self.height] + l[1] + [self.height]
            for j in range(len(line)):
                if line[j] != self.height:
                    line[j] = int(line[j])
            # print(line)
            sim.addDrawingObjectItem(lineContainer, line)

    def set_agent_positions(self, k, info):
        for i in range(k):
            drone = '/Quadcopter['
            obj_path = drone+str(i)+']'
            objHandle = sim.getObject(obj_path)
            print(np.append(info['agent'+str(i+1)],[self.height]))
            x = info['agent'+str(i+1)]
            x = [xi/self.scaling_factor for xi in x]
            x = x + [self.height]
            print(x)
            sim.setObjectPosition(objHandle, -1, x) # Initiate the position of the robots
    
    def set_weed_locations(self, weed_locations):
        weed_obj = sim.getObject('/weed')
        for i, loc in enumerate(weed_locations):
            new_weed_obj = sim.copyPasteObjects([weed_obj])[0]
            x = [xi/self.scaling_factor for xi in loc]
            new_pos = x + [0]
            sim.setObjectPosition(new_weed_obj, -1, new_pos)

    def move_agents(self, k, info):
        for i in range(k):
            obj_path = '/target[' + str(i) + ']'
            objHandle = sim.getObject(obj_path)
            prev_pos = sim.getObjectPosition(objHandle, -1) # current object position
            print(np.append(info['agent'+str(i+1)],[self.height]))
            x = info['agent'+str(i+1)] # Get the x,y from info of gym env
            x = [xi/self.scaling_factor for xi in x] # scale the x,y
            x = x + [self.height] # add the z (height)
            # print(x)
            sim.setObjectPosition(objHandle, -1, x) # Initiate the position of the robots
            # draw the line
            line_data = prev_pos + x
            sim.addDrawingObjectItem(self.trace_line, line_data)

Simulation using trained network

In [12]:
# Make the environment
env = gym.make('ThreeAgentGridworld-v0', render_mode='human')
env.metadata['render_fps'] = 5
obs, info = env.reset()
env.render()
env = env.unwrapped

# Make the simulator object, draw the field, and set agent positions
drone_simulator = Drone_simulator(polygon=env.poly_vertices, scaling_factor=5, height=height)
drone_simulator.draw_field()
drone_simulator.set_agent_positions(k=num_robots, info=info)
drone_simulator.set_weed_locations(weed_locations=env.infected_locations)

[14.   34.    0.35]
[np.float64(2.8), np.float64(6.8), 0.35]
[40.   25.    0.35]
[np.float64(8.0), np.float64(5.0), 0.35]
[20.   40.    0.35]
[np.float64(4.0), np.float64(8.0), 0.35]


In [13]:
# Start simulation
drone_simulator.start_simulation()
terminated, truncated = False, False
total_rewards = 0
while True:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated,  info = env.step(list(action))
    env.render()
    total_rewards += reward
    print(f"Obs: {obs}, Reward: {reward}, terminated: {terminated}, total_rewards: {total_rewards}, action: {action}")
    if terminated or truncated:
        print('terminated:', terminated, 'truncated:', truncated)
        break
    pygame.event.get()
    drone_simulator.move_agents(k=3, info=info) # Simulate
# drone_simulator.stop_simulation()
# env.close()

Program started
Obs: [ 29 694 134   0], Reward: -3, terminated: False, total_rewards: -3, action: [2 0 2]
[14.   33.    0.35]
[39.   25.    0.35]
[20.   39.    0.35]
Obs: [ 90 680 198   0], Reward: -3, terminated: False, total_rewards: -6, action: [2 0 2]
[14.   32.    0.35]
[38.   25.    0.35]
[20.   38.    0.35]
Obs: [419 344 270   0], Reward: -3, terminated: False, total_rewards: -9, action: [2 0 2]
[14.   31.    0.35]
[37.   25.    0.35]
[20.   37.    0.35]
Obs: [483 613 592   0], Reward: -3, terminated: False, total_rewards: -12, action: [2 0 2]
[14.   30.    0.35]
[36.   25.    0.35]
[20.   36.    0.35]
Obs: [812 280 664   2], Reward: 97, terminated: False, total_rewards: 85, action: [2 0 2]
[14.   29.    0.35]
[35.   25.    0.35]
[20.   35.    0.35]
Obs: [ 62  58 162   2], Reward: -3, terminated: False, total_rewards: 82, action: [2 0 2]
[14.   28.    0.35]
[34.   25.    0.35]
[20.   34.    0.35]
Obs: [380 800 231   2], Reward: -3, terminated: False, total_rewards: 79, action: [

In [14]:
drone_simulator.stop_simulation()
env.close()

**Important**: Do not save when closing the scene in CoppeliaSim Simulator.